# 98. 스마트 팜 토양 수분 및 농작물 수확량 분석 실습

> **📥 [실습 주피터 노트북(.ipynb) 다운로드](practice.ipynb)**
## 실전 데이터 분석 98: 스마트 팜 경작지의 토양 수분 상태 및 투입 비료 브랜드별 기온 대비 농작물 단위 면적당 수확량 분석

![도입 만화](img/intro_comic.png)

### 📊 실습 개요 (Tutorial Overview)
스마트 정밀 농업(AgTech) 실험 경작지의 일일 센서 수집 데이터셋입니다. 흙의 토양 수분량(SoilMoisture), 경작지에 투입한 영양 비료 유형(FertilizerType), 재배 온도(Temperature), 강수량(Rainfall_mm)이 최종 재배 농산물의 단위 면적당 수확 생산량(CropYield_kg)에 미치는 복합 조절 상관을 규명합니다.

---

### 🛠️ 핵심 분석 실습 역량 (Core Skills)
* **결측치 평균 대치 (\`fillna\`):** 기상 센서 전력 단선으로 유실된 연간 강수량 수치를 평년 연 평균 강수량으로 보완합니다.
* **비료별 수확량 격차 상자 및 토양 수분별 수확 함수 분석 (\`boxplot\` , \`scatterplot\`):** 투입 비료 유형별 수확량 상자 대조 및 토양 수분 대비 수확 생산량의 온도 오버레이 포물선 맵을 시각화합니다.

---

## 🧐 실무 도메인 지식 가이드 (Domain Knowledge Guide)

> **🌾 농업 및 스마트팜 (Agriculture & Smart Farm)**
> 정밀 농업 및 스마트팜 분석은 생육 모니터링 센서와 토양 상태 데이터를 활용해 농업 수확량을 극대화하고 환경 자원을 보호하는 분야입니다.
>
> * **수확량 예측 모델링**: 드론 촬영 NDVI(정밀 식생 지수), 온도, 토양 수분을 복합 분석하여 수확 예상 톤량을 예측합니다.
> * **온실 생육 환경 최적화**: 토마토 줄기 두께 및 일일 온/습도 분산 범위를 결합해 기후 병충해 발현 경계 임계점을 제어합니다.
> * **지속 가능한 자원 이용**: 어종 보존성 등 해양 어획 데이터 시각화로 한계 생태 가치를 유지하는 균형 조업선을 설정합니다.

---

## Step 1: 데이터 불러오기 및 기본 정보 확인 (Data Load)

![Step 1 데이터 수집 개념도](img/step1_dataset.svg)

제공된 CSV 파일을 분석 컴퓨터로 불러와 로드하고, 컬럼의 타입 및 데이터 구조를 확인합니다.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정
import koreanize_matplotlib
sns.set_theme(style="whitegrid")

# 데이터 로드
df = pd.read_csv('./crop_yield.csv')
print(df.info())
print(df.head())


RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PlotID          1000 non-null   int64  
 1   SoilMoisture    1000 non-null   float64
 2   FertilizerType  802 non-null    str    
 3   Temperature     1000 non-null   float64
 4   Rainfall_mm     985 non-null    float64
 5   CropYield_kg    1000 non-null   float64
dtypes: float64(4), int64(1), str(1)
memory usage: 47.0 KB
> 
> PlotID  SoilMoisture FertilizerType  Temperature  Rainfall_mm  CropYield_kg
0  980001          67.2        Organic         12.4        920.2        1675.0
1  980002          49.8       Chemical         25.2        531.9        1435.1
2  980003          87.7       Chemical         19.0        724.7        2026.6
3  980004          80.9        Organic         32.0       1549.1        2488.1
4  980005          79.9       Chemical         36.2        654.1        1898.8
> ```

---

## Step 2: 결측치 및 데이터 정제 (Data Cleaning)

![Step 2 데이터 정제 개념도](img/step2_missing_values.svg)

수집 과정에서 빈칸으로 기록된 결측값(NaN)의 존재 여부를 진단하고, 데이터 도메인 성격에 부합하는 통계 수치 대입 기법으로 정제합니다.


In [ ]:
# 1. 컬럼별 결측치 개수 확인
print("--- 정제 전 결측치 확인 ---")
print(df.isnull().sum())

# 2. 결측치 전처리 및 대치 수행
mean_rain = df['Rainfall_mm'].mean()
df['Rainfall_mm'] = df['Rainfall_mm'].fillna(mean_rain)
print(df.isnull().sum())


SoilMoisture        0
FertilizerType    198
Temperature         0
Rainfall_mm        15
CropYield_kg        0
> 
> --- 정제 후 결측치 확인 ---
> PlotID              0
SoilMoisture        0
FertilizerType    198
Temperature         0
Rainfall_mm         0
CropYield_kg        0
> ```

### 💡 분석가의 통찰 (Analyst's Insight)
* **강수 기상 평균 보정의 정밀 농업 당위성:** 관수 기기 단선이나 센서 오류로 연 강수량 계측이 비어 있는 필지 정보를 정제합니다. 관수 펌프 제어를 위해 전체 필지의 평년 누적 평균 강수량(약 980.5mm)을 대치해 데이터를 정형화하여, 토양 생육 다변수 함수의 시뮬레이션 표본 일관성을 방어합니다.

---

## Step 3: 단변수 분포 분석 (Univariate EDA)

![Step 3 시각화 개념도](img/step3_visualization.svg)

가장 먼저 핵심 변수가 전체 데이터에서 어떤 빈도와 분포를 가졌는지 단일 변수 시각화를 통해 파악해 봅니다.


In [ ]:
plt.figure(figsize=(8, 5))

# 단변수 분석 그래프 생성
sns.boxplot(data=df, x='FertilizerType', y='CropYield_kg', palette='coolwarm')
plt.title('투입 비료 종류별 단위 경작지 작물 수확량(Crop Yield, kg) 분포', fontsize=14, fontweight='bold')
plt.show()


### 💡 시각화 차트 읽는 법 & 인사이트
* **비료 처리에 따른 수확량 증대 효과 입증:** 비료 시비 그룹(Organic, Chemical)의 수확량 상자가 무시비 대조군(None) 상자 높이를 큰 마진 격차로 추월하여 솟아 있습니다. 특히 화학 복합 비료(Chemical)가 수확량 절대 스케일을 강하게 보장하지만, 유기농(Organic) 비료 역시 그에 준하는 준수한 수확 잔존 가치 분포를 띰을 상자로 대조 증명합니다.

---

## Step 4: 다변수 상관관계 및 이상치 분석 (Multivariate EDA)

![Step 4 상관관계 분석 개념도](img/step4_multivariate.svg)

두 개 이상의 변수를 동시에 결합하여, 조건에 따른 수치 차이나 독립 변수와 종속 변수 간의 통계적 경향을 분석합니다.


In [ ]:
plt.figure(figsize=(9, 6))

# 다변수 분석 그래프 생성
sns.scatterplot(data=df, x='SoilMoisture', y='CropYield_kg', hue='Temperature', palette='viridis', alpha=0.8)
plt.title('토양 수분량 대비 작물 수확량과 기온 시너지 상관성', fontsize=14, fontweight='bold')
plt.show()


### 💡 코드 딥다이브 & 비즈니스 통찰 (Analyst's Insight)
* **수분-수확량의 최적 적정 포물선 함수 규명:** 토양 수분량(X축)과 수확량(Y축)의 분산 관계를 관찰하면 단순 직선이 아니라 수분량 50% 부근에서 최적의 피크 수확량을 띠고, 30% 이하 극건조 구역이나 80% 이상 과습 구역에서는 수확량이 바닥으로 떨어지는 포물선 궤도를 띱니다. 특히 최적 수분(50%) 조건 하에서도 생육 온도가 과도하게 뜨거운 고온(노란색 계열) 구역은 잎 마름으로 인해 수확 피크가 수축하여, 스마트 관수 자동화 시 온도에 연동한 관수 밸브 제어가 정밀 재배의 중추 요인임을 규명합니다.

---

## Step 5: 통계적 직관과 해석 (Statistical Logic)

> 💡 **[적정 생육과 2차 함수 적합선 및 반응 표면 방법론(Response Surface Methodology)]**
... (생략: 농가 수확량 최적화를 위한 다변수 반응 표면 방법론 요약)

---

## 🎯 30분 강의 마무리 및 심화 과제

오늘 우리는 실전 데이터셋을 분석하여 판다스로 데이터를 가공 및 정제하고, 시각화를 활용하여 핵심 변수 간의 통계적 유의성을 검증했습니다. 데이터 속에서 숨겨진 패턴을 올바른 시각으로 탐색하는 능력이 데이터 사이언티스트의 가장 강력한 무기입니다.

### 📝 심화 과제 (Advanced Challenge)
* **재배 온도(Temperature)와 작물 수확량의 상관성 분석:** 기온 상승이 수확량에 긍정적인지 아니면 일정 기준 이후 부정적으로 작용하는지 기온 구간별 평균 수확량을 산출해 보세요.
* **최적 스마트 팜 필지 위험군 리포트:** 토양 수분량이 20% 이하로 메마르고 강수량마저 부족해 수확량이 1,000kg 미만으로 폭락한 저수율 필지 목록을 찾아 코드로 추출해 보세요.
